In [1]:
import sys
sys.argv = ['']  # Jupyter가 넣는 --f 인자 제거

import numpy as np 
import torch
import scipy.sparse as sp

from dataloader import Loader


Current cuda device  0


In [ ]:
def compute_edge_homophily(ori_item_gram_matrix, volume_weight_exponent=1, overlap_weight_exponent=1, \
                              use_volume_weight=True, use_overlap_weight=True):
    """
    Calculate Edge Homophily for symmetric similarity matrix with normalization
    
    Parameters:
    - ori_item_gram_matrix: sp.spmatrix, symmetric item-item similarity matrix
    
    Returns:
    - edge_homophily: float, overall edge homophily
    """
    
    # Calculate item homophily ratio of the item_matrix
    item_gram_matrix_np = ori_item_gram_matrix.toarray()
    
    # Calculate the degree of each item (diagonal of item_gram_matrix)
    item_degree_vector = np.diag(item_gram_matrix_np)

    # Calculate the sum of the degree of two items
    item_degree_sum_mat = np.add.outer(item_degree_vector, item_degree_vector)

    # Compute the homophily ratio for each edge
    # final homophily: w_ij * d_ij / (d_i + d_j - d_ij)
    
    # 1) d_ij / (d_i + d_j - d_ij)
    sim = item_gram_matrix_np / (item_degree_sum_mat - item_gram_matrix_np)

    del item_degree_sum_mat

    # Compute the weight ratio for each edge
    # 2) w_ij = log(1 + d_ij) * d_ij / min(d_i, d_j)
    if use_volume_weight:
        # volume_weight = np.log(1 + item_gram_matrix_np)
        volume_weight = (item_gram_matrix_np) ** volume_weight_exponent
    else:
        volume_weight = 1
    if use_overlap_weight:
        overlap_weight = (item_gram_matrix_np / np.minimum.outer(item_degree_vector, item_degree_vector)) ** overlap_weight_exponent
    else:
        overlap_weight = 1
    total_weight = volume_weight * overlap_weight

    del volume_weight
    del overlap_weight
    
    weight_sim = total_weight * sim

    del sim
    del item_gram_matrix_np
    
    # Get the upper triangle (excluding diagonal)
    upper_tri_weight_sim = np.triu(weight_sim, k=1)
    upper_tri_weight = np.triu(total_weight, k=1)
    
    # Filter out non-zero edges
    edges = upper_tri_weight > 0
    if not np.any(edges):
        return 0.0
    
    # Calculate nominator and denominator for the homophily ratio
    nominator = upper_tri_weight_sim[edges].sum()
    denominator = upper_tri_weight[edges].sum()

    # Compute the final edge homophily
    edge_homophily = nominator / denominator
    
    return edge_homophily

In [3]:
config = {}
config['reg'] = 0.05
config['beta'] = 0.8
config['relax'] = False
config['diag_const'] = False
config['alpha'] = 1
config['alpha_tune'] = False
config['xi'] = 0
config['load'] = None

In [ ]:
datasets = ['ml-20m', 'netflix', 'msd', 'gowalla', 'yelp2018', 'abook']


volume_weight_exponent = 1.5

for dataset in datasets:
    data = Loader("../data/"+dataset)
    all_matrix = sp.vstack([data.UserItemNet,data.validUserItemNet,data.testUserItemNet])
    ori_item_gram_matrix = all_matrix.T.dot(all_matrix)

    edge_homophily = compute_edge_homophily(ori_item_gram_matrix, volume_weight_exponent, use_volume_weight, use_overlap_weight)
    print(dataset, use_volume_weight, use_overlap_weight, edge_homophily)
    
    

loading [../data/ml-20m]
116677 training users, 10000 valid users, 10000 test users, 20108 items
8538846 interactions for training
142514 interactions for validation
139775 interactions for testing
msd is ready to go
ml-20m True True 0.10928365167070453
loading [../data/netflix]
383435 training users, 40000 valid users, 40000 test users, 17769 items
47039694 interactions for training
965461 interactions for validation
970867 interactions for testing
msd is ready to go
netflix True True 0.12652909640717208
loading [../data/msd]
471355 training users, 50000 valid users, 50000 test users, 41140 items
27723767 interactions for training
571324 interactions for validation
572227 interactions for testing
msd is ready to go


In [6]:
import math

def gini_coefficient(values: np.ndarray) -> float:
    """0~1 범위 Gini. values는 비음수."""
    values = np.asarray(values, dtype=float).flatten()
    if values.size == 0:
        return 0.0
    total = values.sum()
    if total == 0:
        return 0.0
    sorted_vals = np.sort(values)
    n = values.size
    cum = np.cumsum(sorted_vals)
    gini = (n + 1 - 2 * (cum.sum() / cum[-1])) / n
    return gini

# item-side gini (train+valid+test) by dataset

datasets = ['ml-20m', 'netflix', 'msd', 'gowalla', 'yelp2018', 'abook']


item_ginis = {}
for dataset in datasets:
    data = Loader("../data/"+dataset)
    all_matrix = sp.vstack([data.UserItemNet, data.validUserItemNet, data.testUserItemNet])
    item_counts = np.array(all_matrix.sum(axis=0)).reshape(-1)
    item_ginis[dataset] = gini_coefficient(item_counts)

    print(dataset, "item_gini", item_ginis[dataset])



loading [../data/ml-20m]
116677 training users, 10000 valid users, 10000 test users, 20108 items
8538846 interactions for training
142514 interactions for validation
139775 interactions for testing
msd is ready to go
ml-20m item_gini 0.9015295431944903
loading [../data/netflix]
383435 training users, 40000 valid users, 40000 test users, 17769 items
47039694 interactions for training
965461 interactions for validation
970867 interactions for testing
msd is ready to go
netflix item_gini 0.8629978201688683
loading [../data/msd]
471355 training users, 50000 valid users, 50000 test users, 41140 items
27723767 interactions for training
571324 interactions for validation
572227 interactions for testing
msd is ready to go
msd item_gini 0.5576650177177449
loading [../data/gowalla]
23858 training users, 3000 valid users, 3000 test users, 40981 items
821316 interactions for training
19230 interactions for validation
19729 interactions for testing
msd is ready to go
gowalla item_gini 0.43583453548